# 01 - Prepare Reactant + Reaction-Class Data

Build compact SFT records for `product_smiles -> reactants + reaction_class` from USPTO-50K.

In [ ]:
!pip install -q -U datasets rdkit tqdm

import json
import random
from pathlib import Path
from typing import Iterable, Optional

from datasets import load_dataset
from rdkit import Chem, RDLogger
from tqdm.auto import tqdm

RDLogger.DisableLog('rdApp.warning')
RDLogger.DisableLog('rdApp.error')

SEED = 3407
DATASET_ID = 'pingzhili/uspto-50k'
WORK_DIR = Path('/content/retro_condition_agent')
DATA_DIR = WORK_DIR / 'data'
DATA_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = DATA_DIR / 'reactants_train.jsonl'
EVAL_PATH = DATA_DIR / 'reactants_eval.jsonl'
TEST_PATH = DATA_DIR / 'reactants_test.jsonl'
ROWS_PATH = DATA_DIR / 'reactants_rows.jsonl'

MAX_ROWS = None
EVAL_FRACTION = 0.05
TEST_FRACTION = 0.05
random.seed(SEED)


In [ ]:
SYSTEM_PROMPT = '''You are a chemistry reaction prediction model.
Return valid compact JSON only.
Predict reactants and reaction class for a single-step synthesis of the target product.
Use canonical SMILES strings when possible.
Do not include conditions or explanations.'''

def canonical_smiles(smiles: Optional[str]) -> Optional[str]:
    if not smiles:
        return None
    try:
        mol = Chem.MolFromSmiles(str(smiles).strip())
        if mol is None:
            return None
        for atom in mol.GetAtoms():
            atom.SetAtomMapNum(0)
        return Chem.MolToSmiles(mol, canonical=True)
    except Exception:
        return None

def canonical_side(value) -> Optional[list[str]]:
    if value is None:
        return None
    parts = value.split('.') if isinstance(value, str) else list(value)
    out = []
    seen = set()
    for part in parts:
        can = canonical_smiles(part)
        if can and can not in seen:
            seen.add(can)
            out.append(can)
    return out or None

def first_value(row: dict, keys: Iterable[str]):
    for key in keys:
        value = row.get(key)
        if value not in (None, '', []):
            return value
    return None

def normalize_row(row: dict, split: str, idx: int) -> Optional[dict]:
    rxn = first_value(row, ['reaction_smiles', 'reaction', 'rxn_smiles'])
    reactants = first_value(row, ['reactants_smiles', 'reactants', 'source'])
    product = first_value(row, ['product_smiles', 'product', 'target'])
    if rxn and '>>' in str(rxn):
        left, right = str(rxn).split('>>', 1)
        reactants = reactants or left
        product = product or right
    product_can = canonical_smiles(product)
    reactants_can = canonical_side(reactants)
    if not product_can or not reactants_can:
        return None
    reaction_class = first_value(row, ['class', 'reaction_class', 'label'])
    return {
        'reaction_id': str(first_value(row, ['reaction_id', 'id']) or f'uspto50k_{split}_{idx}'),
        'product_smiles': product_can,
        'reactants': reactants_can,
        'reaction_class': str(reaction_class) if reaction_class is not None else 'unknown',
        'source_split': split,
    }

def make_record(row: dict) -> dict:
    user = json.dumps({
        'task': 'predict_reactants_and_class',
        'product_smiles': row['product_smiles'],
    }, separators=(',', ':'))
    assistant = json.dumps({
        'reaction_class': row['reaction_class'],
        'reactants': row['reactants'],
    }, separators=(',', ':'))
    return {
        'messages': [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': user},
            {'role': 'assistant', 'content': assistant},
        ],
        'metadata': {'reaction_id': row['reaction_id'], 'source': 'USPTO-50K'},
    }

def write_jsonl(path: Path, records: list[dict]) -> None:
    with path.open('w', encoding='utf-8') as f:
        for record in records:
            f.write(json.dumps(record, ensure_ascii=False) + '\n')


In [ ]:
dataset = load_dataset(DATASET_ID)
rows = []
for split, split_data in dataset.items():
    for idx, item in enumerate(tqdm(split_data, desc=f'normalizing {split}')):
        row = normalize_row(dict(item), split, idx)
        if row is not None:
            rows.append(row)

random.shuffle(rows)
if MAX_ROWS is not None:
    rows = rows[:MAX_ROWS]

n_total = len(rows)
n_test = max(1, int(n_total * TEST_FRACTION))
n_eval = max(1, int(n_total * EVAL_FRACTION))
test_rows = rows[:n_test]
eval_rows = rows[n_test:n_test + n_eval]
train_rows = rows[n_test + n_eval:]

records_train = [make_record(r) for r in train_rows]
records_eval = [make_record(r) for r in eval_rows]
records_test = [make_record(r) for r in test_rows]

write_jsonl(TRAIN_PATH, records_train)
write_jsonl(EVAL_PATH, records_eval)
write_jsonl(TEST_PATH, records_test)
write_jsonl(ROWS_PATH, rows)

print('rows:', n_total)
print('train/eval/test:', len(records_train), len(records_eval), len(records_test))
print('saved:', TRAIN_PATH, EVAL_PATH, TEST_PATH)
print(json.dumps(records_train[0], indent=2)[:2000])
